### Imports

In [ ]:
import os
import numpy as np
import skimage.io as io
import cv2 as cv
import random
import shutil

cwd = os.getcwd()
parent_dir = os.path.abspath(os.path.join(cwd, os.pardir))

# Import OpenSlide
OPENSLIDE_PATH = os.path.join(parent_dir, 'openslide-bin-4.0.0.11-windows-x64\\bin')

if hasattr(os, 'add_dll_directory'):
    # Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide
import xml.etree.cElementTree as ET

### Functions

In [5]:

def parse_xml(anno_path, id_num, factor):
    """Parse XML annotation file and extract coordinates for a specific ID."""
    tree = ET.ElementTree(file=anno_path)
    annolist = {}
    root = tree.getroot()
    
    for annotation in root.iter('Annotation'):
        if annotation.attrib.get('Id') == str(id_num):
            i = 0
            for region in annotation.findall(".//Region"):
                vasc = []
                for vertex in region.findall(".//Vertex"):
                    try:
                        x = float(vertex.attrib.get("X"))
                        y = float(vertex.attrib.get("Y"))
                        vasc.append((int(x / factor), int(y / factor)))
                    except Exception as e:
                        print(f"Error parsing coordinates: {e}")
                        continue
                annolist[i] = vasc
                i += 1

    if len(annolist) == 0:
        anno_filename = os.path.basename(anno_path)
        print(f'There are no annotations for ID {id_num} in {anno_filename}')

    return annolist


def is_mostly_white(image, threshold=0.7):
    """Check if an image is mostly white based on threshold."""
    # Convert the image to grayscale
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    # Apply threshold to identify white pixels
    _, binary = cv.threshold(gray, 220, 255, cv.THRESH_BINARY)
    # Calculate the percentage of white pixels
    white_percentage = np.sum(binary == 255) / binary.size
    return white_percentage >= threshold


def extract_patches_with_padding(wsi, annolist, patch_size, factor, mlevel,
                                 padding_percent=0, overlap_percent=0,
                                 min_anno_overlap=0.7):
    """Extract patches from WSI based on annotations."""
    patches = []
    patch_width, patch_height = patch_size
    padding = int(min(patch_width, patch_height) * (padding_percent / 100))
    overlap = int(min(patch_width, patch_height) * (overlap_percent / 100))
    
    for i, coords in annolist.items():

        # convert annotation coordinates to numpy array
        annotation = np.array(coords, dtype=np.int32)

        # find bounding rectangle for the annotation
        x, y, w, h = cv.boundingRect(annotation)
        
        # calculate patch grid within bounding rectangle
        num_patches_x = max(1, (w - overlap) // (patch_width - overlap))
        num_patches_y = max(1, (h - overlap) // (patch_height - overlap))

        # create a dummy mask of the anotation area to check patch overlap
        anno_mask = np.zeros((h, w), dtype=np.uint8)

        # shift annotation coords relative to bounding box
        shifted_anno = annotation.copy()
        shifted_anno[:, 0] -= x # subtract from all x values
        shifted_anno[:, 1] -= y # subtract from all y values

        # fill annotation mask
        cv.fillPoly(anno_mask, [shifted_anno], 1)

        for y_idx in range(num_patches_y):
            for x_idx in range(num_patches_x):
                # Calculate patch position within bounding box
                patch_x = x_idx * (patch_width - overlap)
                patch_y = y_idx * (patch_height - overlap)

                # Ensure we don't go outside bounding box
                patch_end_x = min(patch_x + patch_width, w)
                patch_end_y = min(patch_y + patch_height, h)

                # extract patch region from annotation mask
                patch_anno_region = anno_mask[patch_y:patch_end_y, patch_x:patch_end_x]

                # calculate overlap percentage
                anno_area = patch_width * patch_height
                overlap_area = np.sum(patch_anno_region)
                overlap_percentage = overlap_area / anno_area if anno_area > 0 else 0

                # skip patch extraction if overlap is insufficient
                if overlap_percentage < min_anno_overlap:
                    continue

                # calculate actual WSI coordinates (with padding)
                actual_x = max(x + patch_x - padding, 0)
                actual_y = max(y + patch_y - padding, 0)
                actual_end_x = min(actual_x + patch_width, wsi.level_dimensions[0][0])
                actual_end_y = min(actual_y + patch_height, wsi.level_dimensions[0][1])
                
                # adjust for padding at edges
                actual_patch_width = actual_end_x - actual_x
                actual_patch_height = actual_end_y - actual_y
                
                # read patch from WSI
                patch_img_rgba = np.asarray(
                    wsi.read_region(
                        (actual_x * factor, actual_y * factor),
                        mlevel,
                        (actual_patch_width, actual_patch_height)
                    )
                )
                patch_img = patch_img_rgba[:, :, :3]
                
                # check if patch is mostly background
                if not is_mostly_white(patch_img):
                    patches.append(patch_img)
    
    return patches


def save_patches(patches, output_dir, slide_name, folder_name, patch_size):
    """Save extracted patches to disk."""
    os.makedirs(output_dir, exist_ok=True)
    patch_w, _ = patch_size
    
    for i, patch in enumerate(patches):
        save_as = os.path.join(
            output_dir, 
            f'{slide_name}_{folder_name}_{patch_w}_patch_position_{i}.tif'
        )
        io.imsave(save_as, patch, check_contrast=False)


def generate_patches_for_slide(wsi_path, xml_path, output_base_dir, 
                               patch_sizes, id_folders, factor, mlevel):
    """Main function to generate patches for a single slide."""
    # Load WSI
    wsi = openslide.OpenSlide(wsi_path)
    slidename = os.path.splitext(os.path.basename(wsi_path))[0]
    
    # Loop through each annotation category
    for i, folder in enumerate(id_folders):
        if folder == 'skip':
            continue
            
        anno_id = i + 1
        slide_dir = os.path.join(output_base_dir, folder, slidename)
        
        # Parse annotations
        annolist = parse_xml(xml_path, anno_id, factor)
        
        # Skip if no annotations found
        if not annolist:
            continue
            
        # Generate patches for each patch size
        for patch_size in patch_sizes:
            patches = extract_patches_with_padding(
                wsi, annolist, patch_size, factor, mlevel,
                padding_percent=0, overlap_percent=0
            )
            
            # Save patches
            patch_size_dir = os.path.join(slide_dir, str(patch_size[0]))
            save_patches(
                patches, patch_size_dir, slidename, folder, patch_size
            )
    
    return slidename

### Execute: extract patches

In [4]:
# CONFIGURATION --------------------------------------------------------
mlevel = 0  # reading the WSI at level 1 (40x)
factor = 2 ** mlevel
patch_sizes = [(48, 48), (256, 256)]

# Set the path to the folder containing WSIs and XML annotations
folder_path = r"D:\DCIS_project\DCIS_annotations\annotated" ## TODO

# Define annotation categories
id_folders = ['id1_epithelium', 'id2_stroma', 'id3_other'] ## TODO
# ----------------------------------------------------------------------

# Create base directories for each annotation type
for folder in id_folders:
    if folder != 'skip':
        os.makedirs(os.path.join(folder_path, folder), exist_ok=True)

# Loop through each .svs file in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith(".svs"):
        
        # Construct paths for slide and annotation files
        wsi_path = os.path.join(folder_path, file_name)
        xml_path = os.path.join(folder_path, f"{os.path.splitext(file_name)[0]}.xml")
        
        # Check if XML file exists
        if not os.path.exists(xml_path):
            print(f"Warning: No XML file found for {file_name}")
            continue
        
        # Generate patches for this slide
        print(f"Processing {file_name}...")
        try:
            slidename = generate_patches_for_slide(
                wsi_path, xml_path, folder_path, 
                patch_sizes, id_folders, factor, mlevel
            )
            print(f"Completed processing {slidename}")
        except Exception as e:
            print(f"Error processing {file_name}: {e}")

#### Here is an example of how the directory structure created looks like after extracting patches:

In [6]:
# path/to/input/folder
# ├── id1_stroma\
# │   └── slide_name\
# │       ├── 48\      ← 48x48 patches
# │       └── 256\     ← 256x256 patches
# ├── id2_epithelium\
# │   └── slide_name\
# │       ├── 48\
# │       └── 256\
# └── id3_other\
#     └── slide_name\
#         ├── 48\
#         └── 256\

### Sampling Patches for Cross Validation

In [139]:
## directory tree layort required:

def sample_images(source_folder,
                patches_folder,
                classification, # epithelium, stroma, other (str)
                size, # either 48 or 256
                total_samples=4200):
    # Get the list of child folders in the source folder (i.e. 2XXXXXX_H&E WSIs)
    child_folders = [f for f in os.listdir(source_folder) if os.path.isdir(os.path.join(source_folder, f))]

    # Equally sample images from each child folder
    num_child_folders = len(child_folders)
    if num_child_folders == 0:
        print("No child folders found.")
        return
    
    images_per_folder = total_samples // num_child_folders
    print(f"Sampling {images_per_folder} images from each of the {num_child_folders} child folders")

    # Create destination folder if it doesn't exist
    test_folder = os.path.join(patches_folder, f'{classification}')
    os.makedirs(patches_folder, exist_ok=True)
    os.makedirs(test_folder, exist_ok=True)
    os.path.join(patches_folder, test_folder)

    images_moved = 0

    # Process each child folder
    for child_folder in child_folders:
        child_folder_path = os.path.join(source_folder, child_folder)
        child_folder_patch_path = os.path.join(child_folder_path, str(size))

        # Get a list of image files in the child folder
        image_files = [f for f in os.listdir(child_folder_patch_path) if os.path.isfile(os.path.join(child_folder_patch_path, f))]

        # If there are fewer files than needed, sample all of them
        images_to_sample = min(images_per_folder, len(image_files))
        images_moved += images_to_sample

        print(f"Sample {images_to_sample} patches from {child_folder}")

        # Randomly sample images
        sampled_images = random.sample(image_files, images_to_sample)

        # Create destination folder
        test_path = os.path.join(test_folder, f'{child_folder}', str(size))
        os.makedirs(test_path, exist_ok=True)
        # Copy the sample images to the destination folder

        for image in sampled_images:
            src_image_path = os.path.join(child_folder_patch_path, image)
            test_image_path = os.path.join(test_path, image)

            # Check if the file already exists in the destination folder
            if os.path.exists(test_image_path):
                name, ext = os.path.splitext(image)
                test_image_path = os.path.join(test_folder, f"{name}_{child_folder}{ext}")

            shutil.copy(src_image_path, test_image_path)
    print(f"Successfully sampled and copied {images_moved} images.")

In [140]:
# TODO
source_folder = r'D:\DCIS_project\3CC_training\annotations\annotated\id1_epithelium'
training_folder = r'D:\DCIS_project\3CC_training\annotations\annotated\test_patches'
size = 256 # 48 or 256
total_samples = 4 # default is 4200
classification = 'epithelium'

sample_images(source_folder,training_folder,classification, size, total_samples)

Sampling 2 images from each of the 2 child folders
Sample 2 patches from 102_A6.1_H&E
Sample 2 patches from 84_D12.1_H&E
Successfully sampled and copied 4 images.


### Adjusting Contrast, Brightness, and Saturation on Patches

In [33]:
def adjust_img(image_folder, image_dist, contrast=1, brightness=1, saturation=1):
    # create destination folder if it doesn't exist
    os.makedirs(image_dist, exist_ok=True)
    imgs = os.listdir(image_folder)

    if len(imgs) == 0:
        print(f"No images found in {image_folder}")
        return
    print(f'Processing {len(imgs)} images in {image_folder}...')

    for img_name in imgs:
        img_path = os.path.join(image_folder, img_name)
        img = cv.imread(img_path)

        if img is None:
            continue
        # adjusting saturation
        hsv_image = cv.cvtColor(img, cv.COLOR_BGR2HSV)
        h, s, v = cv.split(hsv_image)
        s = s.astype('float32') * saturation
        s = s.astype("uint8")
        hsv_adjusted = cv.merge([h, s, v])

        sat_adjusted_img = cv.cvtColor(hsv_adjusted, cv.COLOR_HSV2BGR)

        # adjusting contrast and brightness
        final_adjusted_img = cv.convertScaleAbs(sat_adjusted_img, alpha=contrast, beta=brightness)

        save_path = os.path.join(image_dist, img_name)
        cv.imwrite(save_path, final_adjusted_img)
    print(f'Successful adjusted {len(imgs)} images')

    return

In [115]:
# TODO 
image_folder = 'd:\DCIS_project\\3cc_sat_contrast_run\og_patches_125' # folder of images to adjust
image_dist = 'D:\DCIS_project\\3cc_sat_contrast_run\\3CC_model_2.h5\\test_run_3\\125_A10.1_H&E\patches' # folder to place adjusted images
contrast = 0.8 # default is 1.0
# brightness = '' # default is 1.0
saturation = 1.4 # default is 1.0

adjust_img(image_folder, image_dist, contrast=contrast, saturation=saturation)

Processing 6 images in d:\DCIS_project\3cc_sat_contrast_run\og_patches_125...
Successful adjusted 6 images


### Overlaying Images

In [88]:
def overlay(folder1, folder2, alpha, beta, file_dst, gamma=0):
    '''
    To overlay two images
    INPUTS
    folder1 - (str) folder of H&E images
    folder2 - (str) folder of 3cc classification images
    alpha - (float) opacity of first image
    beta - (float) opacity of second image
    gamma - (float) scalar added to each sum - default=0
    file_dst - (str) folder created (if doesn't exist) to place overlayed images
    '''
    # creat destination folder if it doesn not exist
    os.makedirs(file_dst, exist_ok=True)

    # ensure both folders have exact matching image names

    
    imgs1 = os.listdir(folder1)
    imgs2 = os.listdir(folder2)
    print(f'Found {len(imgs1)} images in {folder1} and {len(imgs2)} images in {folder2}')
    for img_name in imgs1:
        path1 = os.path.join(folder1, img_name)
        path2 = os.path.join(folder2, f'Classified_{img_name}') # assuming classification images are named "Classified_{original_image_name}" like in 3CC

        img1 = cv.imread(path1)
        img2 = cv.imread(path2)

        # check if images were loaded successfully
        if img1 is None or img2 is None:
            print(f"Error loading images: {img_name}")
            continue

        # make sure images are the same size
        img2 = cv.resize(img2, (img1.shape[1], img1.shape[0]))

        # overlay images
        overlayed_img = cv.addWeighted(img1, alpha, img2, beta, gamma)
        imgname = f'overlaid_{img_name}'
        save_path = os.path.join(file_dst, imgname)
        cv.imwrite(save_path, overlayed_img)
    print(f'Successfully overlayed {len(imgs1)} images and saved to {file_dst}')
    return


In [136]:
# TODO
folder1 = 'D:\DCIS_project\\3cc_sat_contrast_run\og_patches_125' # folder of H&E images
folder2 = 'd:\DCIS_project\\3cc_sat_contrast_run\\3CC_model_2.h5\\test_run_3\\125_A10.1_H&E\\3class' # folder of 3cc classification images
alpha = 0.85 # opacity of H&E image 0.0-1.0
beta = 1 - alpha # opacity of classification image
file_dst = 'D:\DCIS_project\\3cc_sat_contrast_run\\3CC_model_2.h5\\test_run_3\\125_A10.1_H&E\overlaid' # folder to place overlayed images

overlay(folder1, folder2, alpha, beta, file_dst)

Found 6 images in D:\DCIS_project\3cc_sat_contrast_run\og_patches_125 and 6 images in d:\DCIS_project\3cc_sat_contrast_run\3CC_model_2.h5\test_run_3\125_A10.1_H&E\3class
Successfully overlayed 6 images and saved to D:\DCIS_project\3cc_sat_contrast_run\3CC_model_2.h5\test_run_3\125_A10.1_H&E\overlaid


### Choose patches for retraining [OLD]

In [8]:
## directory tree layout required for this:

def sample_images(source_folder, 
                  test_folder,
                  size, # either 48 or 256
                  total_samples=4200):
    # Get the list of child folders (i.e. 2XXXXXX_H&E)
    child_folders = [f for f in os.listdir(source_folder) if os.path.isdir(os.path.join(source_folder, f))]

    # Calculate how many images to sample from each child folder
    num_child_folders = len(child_folders)
    if num_child_folders == 0:
        print("No child folders found.")
        return

    images_per_folder = total_samples // num_child_folders
    print(f"Sampling {images_per_folder} images from each of the {num_child_folders} child folders.")

    # Ensure the destination folder exists
    os.makedirs(test_folder, exist_ok=True)

    images_moved = 0

    # Process each child folder
    for child_folder in child_folders:
        child_folder_path = os.path.join(source_folder, child_folder)
        child_folder_patch_path = os.path.join(child_folder_path, str(size))

        # Get a list of image files in the child folder
        image_files = [f for f in os.listdir(child_folder_patch_path) if os.path.isfile(os.path.join(child_folder_patch_path, f))]

        # If there are fewer files than needed, sample all of them
        images_to_sample = min(images_per_folder, len(image_files))
        images_moved += images_to_sample

        print(f"sampling {images_to_sample} patches from {child_folder}")

        # Randomly sample images
        sampled_images = random.sample(image_files, images_to_sample)

        # Copy the sampled images to the destination folder
        for image in sampled_images:
            src_image_path = os.path.join(child_folder_patch_path, image)
            test_image_path = os.path.join(test_folder, image)

            # Check if the file already exists in the destination folder to avoid overwriting
            if os.path.exists(test_image_path):
                name, ext = os.path.splitext(image)
                test_image_path = os.path.join(test_folder, f"{name}_{child_folder}{ext}")

            shutil.copy(src_image_path, test_image_path)
            # print(f"Copied {image} to {test_image_path}")

    print(f"Successfully sampled and copied {images_moved} images.")

In [10]:
# TODO
source_folder = "D:\DCIS_project\DCIS_annotations\\annotated\id1_epithelium"  
training_folder = "D:\DCIS_project\DCIS_annotations\\annotated\epithelium" 
size = 48
total_samples = 100

sample_images(source_folder, training_folder, size, total_samples)

Sampling 50 images from each of the 2 child folders.
sampling 50 patches from 102_A6.1_H&E
sampling 50 patches from 84_D12.1_H&E
Successfully sampled and copied 100 images.
